# GENA_LM Valid14 Inference Using Official Dataset

This notebook uses the same `ExpressionDataset` + model input format as `run_expression_finetuning_final.py`. This is the safest notebook version for checking notebook inference against bash inference.

In [1]:
# 1. Paths and settings
from pathlib import Path

TASK_ROOT = Path("/mnt/newdata/dpanc/benchmarking/GENA_LM")
REPO = TASK_ROOT / "GENA_LM_expression_branch"
CONFIG_PATH = TASK_ROOT / "GENA_LM/downstream_tasks/expression_prediction/configs/final_inference_valid_14.yaml"
OUT_DIR = TASK_ROOT / "notebook_inference_valid14_official"
OUT_CSV = OUT_DIR / "valid14_official_dataset_predictions.csv"

# Keep 2 for fast smoke test. Set to None for all valid genes.
TEST_N_GENES = 2

print("CONFIG_PATH:", CONFIG_PATH)
print("OUT_CSV:", OUT_CSV)
print("TEST_N_GENES:", TEST_N_GENES)


CONFIG_PATH: /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM/downstream_tasks/expression_prediction/configs/final_inference_valid_14.yaml
OUT_CSV: /mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid14_official/valid14_official_dataset_predictions.csv
TEST_N_GENES: 2


In [2]:
# 2. Imports and environment
import os
import sys
from itertools import chain

import numpy as np
import pandas as pd
import torch
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf
from transformers import AutoTokenizer

os.environ["GENALM_HOME"] = str(TASK_ROOT)
os.environ["TMPDIR"] = str(TASK_ROOT / "cache/tmp")
os.environ["TRITON_CACHE_DIR"] = str(TASK_ROOT / "cache/triton")
os.environ["TORCHINDUCTOR_CACHE_DIR"] = str(TASK_ROOT / "cache/torchinductor")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
for p in [OUT_DIR, Path(os.environ["TMPDIR"]), Path(os.environ["TRITON_CACHE_DIR"]), Path(os.environ["TORCHINDUCTOR_CACHE_DIR"] )]:
    p.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))

from downstream_tasks.expression_prediction.expression_model_final import ExpressionCounts

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


/home/dpanc/benchmarking/GENA_LM/envs/expression_flash/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.4.0
cuda: True
gpu: NVIDIA A100 80GB PCIe


In [3]:
# 3. Load YAML config, model, and official valid dataset
with initialize_config_dir(str(CONFIG_PATH.parent), version_base=None):
    cfg = compose(config_name=CONFIG_PATH.name)

args = instantiate(cfg["args_params"])
model_kwargs = instantiate(cfg["model_kwargs"])
model = ExpressionCounts(**model_kwargs)
model.load_state_dict(torch.load(args["init_checkpoint"], map_location="cpu", weights_only=True))
model.eval()

# Merge shared_dataset_params into valid_dataset config exactly like the runner does.
def merge_default_params_with_dataset_config(dataset_config, default_params):
    merged = OmegaConf.create(OmegaConf.to_container(dataset_config, resolve=True))
    defaults = OmegaConf.to_container(default_params, resolve=True)

    def rec(target, source):
        for key, value in source.items():
            if key in target:
                if isinstance(value, dict) and isinstance(target[key], dict):
                    rec(target[key], value)
            else:
                target[key] = value.copy() if isinstance(value, dict) else value

    rec(merged, defaults)
    return merged

valid_cfg = merge_default_params_with_dataset_config(
    cfg["valid_dataset_human_valid_14"],
    cfg["shared_dataset_params"],
)
OmegaConf.update(valid_cfg, "n_keys", 14, force_add=True)
valid_dataset = instantiate(valid_cfg)

print("Dataset length:", len(valid_dataset))
print("n_keys:", valid_dataset.n_keys)


Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


Using ModernGENA from /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM/models/modernbert_large
missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []
bert dropouts: {'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'mlp_dropout': 0.1}


qwen dropouts: {'attention_dropout': 0.1}
[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - layers.25.self_attn.q_proj.weight
  - layers.25.self_attn.k_proj.weight
  - layers.25.self_attn.v_proj.weight
  - layers.25.self_attn.o_proj.weight
  - layers.25.self_attn.q_norm.weight
  - layers.25.self_attn.k_norm.weight
  - layers.25.mlp.gate_proj.weight
  - layers.25.mlp.up_proj.weight
  - layers.2

Dataset length: 3038
n_keys: 14


In [4]:
# 4. Official-like collate function for batch_size=1
# This copies the important padding behavior from run_expression_finetuning_final.py.

dna_tokenizer = AutoTokenizer.from_pretrained(args["gen_tokenizer"], trust_remote_code=True)
text_tokenizer = AutoTokenizer.from_pretrained(args["text_tokenizer"], trust_remote_code=True)

def pad_1d(x, length, value, pad_left=False):
    pad_len = length - x.size(0)
    if pad_len <= 0:
        return x
    pad = x.new_full((pad_len,), value)
    return torch.cat([pad, x], dim=0) if pad_left else torch.cat([x, pad], dim=0)

def pad_2d(x, max_len, value, dim=1):
    pad_len = max_len - x.size(dim)
    if pad_len <= 0:
        return x
    shape = list(x.shape)
    shape[dim] = pad_len
    pad = x.new_full(tuple(shape), value)
    return torch.cat([x, pad], dim=dim)

def pad_3d(x, max_len, value, dim=1):
    pad_len = max_len - x.size(dim)
    if pad_len <= 0:
        return x
    shape = list(x.shape)
    shape[dim] = pad_len
    pad = x.new_full(tuple(shape), value)
    return torch.cat([x, pad], dim=dim)

def collate_one(samples):
    pad_keys = ["input_ids", "attention_mask", "token_type_ids", "labels", "labels_mask"]
    no_pad_keys = ["dataset_flag"]
    special_keys = ["gene_id", "selected_keys", "dataset_description"]

    pad_ids = {
        "input_ids": dna_tokenizer.pad_token_id,
        "attention_mask": 0,
        "token_type_ids": 0,
        "labels": 0.0,
        "labels_mask": 0,
        "desc_input_ids": text_tokenizer.pad_token_id,
        "desc_attention_mask": 0,
    }

    max_seq_len = max(sample["input_ids"].size(1) for sample in samples)
    n_keys = len(samples[0]["desc_input_ids"])
    max_text_len = max(ids.size(0) for sample in samples for ids in sample["desc_input_ids"])

    batch = {key: [] for key in pad_keys + no_pad_keys + special_keys}
    desc_ids_batch = []
    desc_mask_batch = []

    for sample in samples:
        desc_ids_batch.append(torch.stack([
            pad_1d(sample["desc_input_ids"][k], max_text_len, pad_ids["desc_input_ids"], pad_left=True)
            for k in range(n_keys)
        ]))
        desc_mask_batch.append(torch.stack([
            pad_1d(sample["desc_attention_mask"][k], max_text_len, 0, pad_left=True)
            for k in range(n_keys)
        ]))

    for sample in samples:
        for key in pad_keys:
            x = sample[key]
            if key in ["input_ids", "attention_mask", "token_type_ids"]:
                x = pad_2d(x, max_seq_len, pad_ids[key], dim=1)
            else:
                x = pad_3d(x, max_seq_len, pad_ids[key], dim=1)
            batch[key].append(x)

        for key in no_pad_keys:
            batch[key].append(sample[key])

        for key in special_keys:
            batch[key].append(sample[key])

    for key in pad_keys + no_pad_keys:
        batch[key] = torch.stack(batch[key], dim=0)

    batch["desc_input_ids"] = torch.stack(desc_ids_batch, dim=0)
    batch["desc_attention_mask"] = torch.stack(desc_mask_batch, dim=0)

    return batch


In [5]:
# 5. Run inference
# Output is collected in long form, then pivoted to gene_id x cell_type.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

n_items = len(valid_dataset) if TEST_N_GENES is None else min(TEST_N_GENES, len(valid_dataset))
rows = []

tensor_keys = [
    "input_ids", "attention_mask", "labels", "labels_mask",
    "dataset_flag", "desc_input_ids", "desc_attention_mask",
]

for item_idx in range(n_items):
    batch = collate_one([valid_dataset[item_idx]])
    model_batch = {
        key: (value.to(device) if key in tensor_keys else value)
        for key, value in batch.items()
    }

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16), torch.no_grad():
        output = model(
            input_ids=model_batch["input_ids"],
            attention_mask=model_batch["attention_mask"],
            labels=model_batch["labels"],
            labels_mask=model_batch["labels_mask"],
            desc_input_ids=model_batch["desc_input_ids"],
            desc_attention_mask=model_batch["desc_attention_mask"],
            dataset_flag=model_batch["dataset_flag"],
        )

    logits = output["logits"].detach().cpu().float()
    masks = output["labels_mask_reshaped"].detach().cpu()

    y_pred = logits[:, 0, 0][masks[:, 0, 0] > 0].numpy()
    gene_ids = list(chain.from_iterable(batch["gene_id"]))
    cell_ids = list(chain.from_iterable(batch["selected_keys"]))

    for gene_id, cell_id, pred in zip(gene_ids, cell_ids, y_pred):
        rows.append({"gene_id": gene_id, "cell_type": cell_id, "prediction": float(pred)})

    if item_idx < 5 or (item_idx + 1) % 100 == 0:
        print(f"Processed {item_idx + 1}/{n_items}")

pred_long = pd.DataFrame(rows)
pred_matrix = pred_long.pivot_table(index="gene_id", columns="cell_type", values="prediction", aggfunc="first")
pred_matrix = pred_matrix.reset_index()

print("pred_matrix shape:", pred_matrix.shape)
display(pred_matrix.head())


Processed 1/2
Processed 2/2
pred_matrix shape: (2, 15)


cell_type,gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
0,ENSG00000143942.4,1.226562,0.75000,0.746094,1.0625,0.773438,0.644531,1.320312,1.140625,1.250000,1.218750,0.906250,1.757812,1.085938,0.714844
1,ENSG00000232604.1,-0.001007,0.00528,-0.027344,-0.0271,-0.025391,-0.000074,-0.024048,-0.015747,-0.001984,0.003662,-0.010864,0.001968,-0.012024,0.005859


In [6]:
# 6. Save predictions
pred_matrix.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)


Saved: /mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid14_official/valid14_official_dataset_predictions.csv


In [7]:
# 7. Optional sanity check against bash predictions for overlapping genes
bash_pred_path = TASK_ROOT / "runs/expression/final_inference_valid_14/20260706-175225/Expression_dataset_v1_GRCh38_csv_valid14 dataset_pred.csv"

if bash_pred_path.exists():
    bash = pd.read_csv(bash_pred_path).set_index("gene_id")
    nb = pred_matrix.set_index("gene_id")
    common_genes = nb.index.intersection(bash.index)
    common_cells = nb.columns.intersection(bash.columns)
    max_abs_diff = np.nanmax(np.abs(nb.loc[common_genes, common_cells].to_numpy() - bash.loc[common_genes, common_cells].to_numpy()))
    print("Compared with:", bash_pred_path)
    print("common genes:", len(common_genes))
    print("common cells:", len(common_cells))
    print("max abs diff:", max_abs_diff)


Compared with: /mnt/newdata/dpanc/benchmarking/GENA_LM/runs/expression/final_inference_valid_14/20260706-175225/Expression_dataset_v1_GRCh38_csv_valid14 dataset_pred.csv
common genes: 2
common cells: 14
max abs diff: 0.0
